### Import bibliotek i konfiguracja

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 1
np.random.seed(RANDOM_STATE)

ADOPTION_SPEED_LABELS = {
    0: '0–7',
    1: '8–30',
    2: '31–90',
    3: '>100',
}
CLASS_ORDER = sorted(ADOPTION_SPEED_LABELS.keys())
CLASS_NAMES = [ADOPTION_SPEED_LABELS[c] for c in CLASS_ORDER]

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### Wczytanie danych i modeli

In [3]:
PROCESSED_DIR = '../data/processed'
MODELS_DIR = Path('../models')
INPUT_PATH = f'{PROCESSED_DIR}/train_clean_anomaly.csv'

df = pd.read_csv(INPUT_PATH)
if 'is_anomaly' in df.columns:
    df = df.drop(columns=['is_anomaly'])

y = df['AdoptionSpeed']
X = df.drop(columns=['AdoptionSpeed'])

# Wczytanie modeli i listy cech
rf_model = joblib.load(MODELS_DIR / 'rf_best.pkl')
svm_model = joblib.load(MODELS_DIR / 'svm_best.pkl')
knn_model = joblib.load(MODELS_DIR / 'knn_best.pkl')
nn_model = joblib.load(MODELS_DIR / 'nn_best.pkl')
feature_names = joblib.load(MODELS_DIR / 'feature_names.pkl')

X = X[feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Skalowanie (dla SVM/KNN/MLP)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

# RF - dane nieskalowane
base_acc = {
    'Random Forest': accuracy_score(y_test, rf_model.predict(X_test)),
    'SVM': accuracy_score(y_test, svm_model.predict(X_test_scaled)),
    'KNN': accuracy_score(y_test, knn_model.predict(X_test_scaled)),
    'MLP': accuracy_score(y_test, nn_model.predict(X_test_scaled)),
}

print('\nAccuracy każdego z modeli')
display(pd.DataFrame({'accuracy': base_acc}).round(4))

X_train: (11875, 20), X_test: (2969, 20)

Accuracy każdego z modeli


,accuracy
Random Forest,0.4160
SVM,0.3907
KNN,0.3654
MLP,0.4015


### Hard Voting

### Weighted Voting